# Week 10 (Live) — Multi-Agent System with LangGraph (Student)

## 0. Setup

In [37]:
# HINT: %pip install -q langgraph langchain-openai langchain-tavily python-dotenv

import os

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(), override=True)

os.getenv("OPENAI_API_KEY")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [38]:
llm.invoke("Hello")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_88876bec1e', 'id': 'chatcmpl-DyHRDfkxBZMNt4hCozh7ktF5G1YSy', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f3286-3183-7423-a380-c6d9014c91a7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## 1. Shared State

Every node in the graph reads from and writes to the same **state object.** 

- `query`
- `research_data`
- `final_output`

 state = { }

In [39]:
# HINT: from typing import TypedDict
# HINT: class AgentState(TypedDict): define exactly 3 fields ->
#       query: str          (the original user question)
#       research_data: str  (facts found by the Research Agent, "" if skipped)
#       final_output: str   (the finished answer from the Writer Agent)

from typing import TypedDict  # Graph state is simply a dictionary

class AgentState(TypedDict):  # Wedefine exactly what keys and state must contain, this defines the Shared memory for AI agents
    query: str
    research_data: str
    final_output: str

## 2. The Search Tool (Research Agent's tool)

In [40]:
# HINT: from langchain_core.tools import tool

# HINT: create MOCK_KNOWLEDGE_BASE - a dict of {topic: [list of fact strings]}
#       for 1-2 topics you expect to ask about

# HINT: write _mock_search(query) ->
#       - lowercase the query
#       - check which topics have any word overlapping with the query
#       - collect and de-duplicate the matching facts
#       - return them as a "\n"-joined bullet list (fall back to a friendly
#         "no mock data" message if nothing matched)

# HINT: check os.getenv("TAVILY_API_KEY") -> if present, try:
#       from langchain_tavily import TavilySearch
#       and create a client with max_results=5 (wrap in try/except ImportError)

# HINT: write an @tool-decorated function web_search_tool(query) that:
#       - if a real Tavily client was created, call it and pull out each
#         result's "content" field
#       - otherwise, fall back to _mock_search(query)

# HINT: test it - print(web_search_tool.invoke("some test query"))


from langchain_core.tools import tool # Any Python function can be converted to a tool using @tool decorator;

# MOCK_KNOWLEDGE_BASE = {
#     "2008 financial crisis": [
#         "The crisis was triggered by the collapse of the U.S. housing bubble and a wave of subprime mortgage defaults.",
#         "Banks had bundled risky subprime mortgages into complex securities (MBS/CDOs) that were rated much safer than they actually were.",
#         "Excessive leverage and risk-taking by large financial institutions amplified the losses when housing prices fell.",
#         "The bankruptcy of Lehman Brothers in September 2008 triggered a global panic and froze credit markets.",
#         "Governments responded with bailouts (e.g., TARP in the U.S.) and central banks slashed interest rates to stabilize the system.",
#     ],
#     "small business": [
#         "Credit markets froze in 2008-2009, making it far harder for small businesses to get loans or lines of credit.",
#         "Consumer spending dropped sharply during the recession, cutting revenue for small, locally-owned businesses.",
#         "Many small businesses were forced to lay off employees or shut down entirely between 2008 and 2010.",
#         "U.S. unemployment rose to about 10% by late 2009, further reducing demand for small-business goods and services.",
#         "SBA-backed small business loan volume fell significantly in 2008-2009 as banks tightened lending standards.",
#     ],
# }


# def _mock_search(query: str) -> str:     # internal helper function
#     """Fallback search: look up canned facts from the local MOCK_KNOWLEDGE_BASE."""
#     query_lower = query.lower()
#     hits = []
#     for topic, facts in MOCK_KNOWLEDGE_BASE.items():
#         # very simple keyword match: if any word from the topic appears in the query, include its facts
#         if any(word in query_lower for word in topic.split()):
#             hits.extend(facts)

#     if not hits:
#         hits = [f"No mock data available for '{query}'. (A real search tool would fetch live results here.)"]

#     # de-duplicate while preserving order, then format as bullet points
#     unique_hits = list(dict.fromkeys(hits))
#     return "\n".join(f"- {fact}" for fact in unique_hits)

_tavily_client = None
if os.getenv("TAVILY_API_KEY"):
    try:
        from langchain_tavily import TavilySearch
        _tavily_client = TavilySearch(max_results=5)
        print("web_search_tool: using REAL Tavily search (TAVILY_API_KEY found).")
    except ImportError:
        print("web_search_tool: TAVILY_API_KEY found but langchain-tavily isn't installed - using mock search.")
else:
    print("web_search_tool: no TAVILY_API_KEY found - using mock search.")

@tool
def web_search_tool(query: str) -> str:
    """Search the web for facts relevant to the query and return short bullet-point findings."""

    if _tavily_client is not None:
        response = _tavily_client.invoke({"query": query})
        results = response.get("results", []) if isinstance(response, dict) else []
        if results:
            # Each Tavily result has a "content" snippet - that's the fact we care about.
            return "\n".join(f"- {item['content']}" for item in results if item.get("content"))

    # Fall back to the mock knowledge base if Tavily isn't set up or returned nothing.
    return _mock_search(query)

# Quick manual test of the tool by itself
print(web_search_tool.invoke("2008 financial crisis small business"))


web_search_tool: using REAL Tavily search (TAVILY_API_KEY found).
- But for small businesses, limiting their access to capital when they need it most can create a negative feedback loop – as they struggle, and in some cases even shutter, the economy further weakens, which makes banks even more cautious. We can look to important lessons from the financial crisis of 2008 to help community banks provide the support that will help communities survive the hardship and re-emerge in a new normal. Millions of businesses kept the lights on in 2020 with help from Paycheck Protection Program loans offered through the Small Business Administration as part of the CARES Act. A second round of PPP loans will be offered, which is long-awaited good news for small businesses, but these loans aren’t a panacea for all of the financial hardship facing small business owners right now. Private community lenders can be the hero that small businesses need by extending credit to the millions of small businesses

Research Agent -> Web Search Tool -> Tavily AVailable -> Search Internet

If the API key is not avilable or its failing -> Mock Search

## 3. Supervisor Node

In [41]:
# HINT: define CONVERSATIONAL_PATTERNS - a list of greeting/small-talk words
#       like "hi", "hello", "thanks", "how are you"

# HINT: def supervisor_node(state):
#       - print a "Thought" message showing the query it received
#       - return {} (it does NOT change any state - it only routes)

# HINT: def route_after_supervisor(state) -> str:
#       - lowercase state["query"]
#       - check if it matches any CONVERSATIONAL_PATTERNS
#       - print your routing decision
#       - return "writer_agent" if conversational, else "research_agent"


# Research/ Writer Agent
CONVERSATIONAL_PATTERNS = ["hi", "hello", "hey", "thanks", "thank you", "how are you", "who are you"]

def supervisor_node(state: AgentState) -> dict:
    print(f"[SUPERVISOR] Thought: I received the query -> '{state['query']}'."
          f"I will decide wheather this needs research, but I will not answer it myself.")

    return {}

def route_after_supervisor(state: AgentState) -> str:
    """Conditional edge function: returns the NAME of the next node to run"""
    query_lower = state['query'].lower()
    needs_research = not any(pattern in query_lower for pattern in CONVERSATIONAL_PATTERNS)

    if needs_research:
        print("[SUPERVISOR] Decision: this looks like a factual question -> route to Research Agent")
        return "research_agent"
    else:
        print("[SUPERVISOR] Decision: this looks conversational -> skip research, route to Writer agent")
        return "writer_agent"

## 4. Research Agent Node

Call the search tool and record what it found;

In [42]:
# HINT: def research_agent_node(state):
#       - print a "Thought" message using state["query"]
#       - call web_search_tool.invoke(state["query"])
#       - print which tool was called and what it returned
#       - return {"research_data": <the tool's result>}

def research_agent_node(state: AgentState) -> dict:
    print(f"[RESEARCH AGENT] Thought: I need facts about -> '{state['query']}'. Calling the web_search tool")

    tool_result = web_search_tool.invoke(state['query'])

    print(f"[RESEARCH AGENT] Tool called: web_search_tool")
    print(f"[RESEARCH AGENT] Tool return: \n{tool_result}")

    return {"research_data": tool_result}



## 5. Writer Agent Node

In [43]:
# HINT: def writer_agent_node(state):
#       - has_research = bool(state["research_data"].strip())
#       - print a "Thought" message
#       - if has_research: build a prompt that says "use ONLY the research
#         notes below" and includes state["research_data"] + state["query"]
#       - else: build a simpler prompt using just state["query"]
#       - call llm.invoke(prompt)
#       - print that the LLM produced a final_output (and its length)
#       - return {"final_output": response.content}
def writer_agent_node(state: AgentState) -> dict:
    has_research = bool(state['research_data'].strip())
    print(f"[WRITER AGENT] Thought: I will now compose the final answer "
              f"({'using the research findings' if has_research else 'directly, since no research was needed'}).")
    
    if has_research:
        prompt = (
                    "You are a helpful writing assistant. Using ONLY the research notes below, write a "
                    "clear, well-organized answer to the user's question. Use short paragraphs or bullet "
                    "points where helpful.\n\n"
                    f"Research notes:\n{state['research_data']}\n\n"
                    f"Question: {state['query']}\n\nAnswer:"
                )
    else:
        prompt = f"Answer this question clearly and concisely: \n\n{state['query']}"

    response = llm.invoke(prompt)
    print("[WRITER AGENT] Tool called: none (this node calls the LLM directly, not a tool)")
    print(f"[WRITER AGENT] LLM produced final_output ({len(response.content)} characters).")
    
    return {"final_output": response.content}


## 6. Wire It Together: Build and Compile the Graph

In [44]:
# HINT: from langgraph.graph import StateGraph, START, END
# HINT: builder = StateGraph(AgentState)

# HINT: builder.add_node(...) for "supervisor", "research_agent", "writer_agent"

# HINT: builder.add_edge(START, "supervisor")

# HINT: builder.add_conditional_edges(
#           "supervisor",
#           route_after_supervisor,
#           {"research_agent": "research_agent", "writer_agent": "writer_agent"},
#       )

# HINT: builder.add_edge("research_agent", "writer_agent")
# HINT: builder.add_edge("writer_agent", END)

# HINT: graph = builder.compile()

# HINT: try/except: print(graph.get_graph().draw_mermaid())


from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState) # This creates the graph builder that knows the shape of our shared state.

# Nodes:
builder.add_node("supervisor", supervisor_node)        # add_node registers each function under a string name
builder.add_node("research_agent", research_agent_node)
builder.add_node("writer_agent", writer_agent_node)

# Edges:
builder.add_edge(START, "supervisor")

# Conditional edge:
builder.add_conditional_edges(
    "supervisor",
    route_after_supervisor,
    {
        "research_agent": "research_agent",
        "writer_agent": "writer_agent"
    },
)

builder.add_edge("research_agent", "writer_agent")
builder.add_edge("writer_agent", END)

graph = builder.compile()

#### Lets Visualize this graph structure

In [45]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	supervisor(supervisor)
	research_agent(research_agent)
	writer_agent(writer_agent)
	__end__([<p>__end__</p>]):::last
	__start__ --> supervisor;
	research_agent --> writer_agent;
	supervisor -.-> research_agent;
	supervisor -.-> writer_agent;
	writer_agent --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 7. Run It

In [46]:
# HINT: build initial_state matching AgentState:
#       - query = your own example question
#       - research_data = ""
#       - final_output = ""

# HINT: result = graph.invoke(initial_state)

# HINT: print result["final_output"]

# TRY THIS: change the query to something conversational (like "hi") and
# re-run - which path does the Supervisor take this time?
if __name__ == "__main__":
    example_query = "What is the effect of the demonitisation in our economy how did it affect the small businesses"

    inital_state: AgentState = {
        "query": example_query,
        "research_data": "",
        "final_output": "",
    }

    print("USER QUERY:", example_query)
    print("==="*50)

    result = graph.invoke(inital_state)

    print("==="*50)
    print("FINAL ANSWER")
    print("==="*50)
    print(result['final_output'])

USER QUERY: What is the effect of the demonitisation in our economy how did it affect the small businesses
[SUPERVISOR] Thought: I received the query -> 'What is the effect of the demonitisation in our economy how did it affect the small businesses'.I will decide wheather this needs research, but I will not answer it myself.
[SUPERVISOR] Decision: this looks like a factual question -> route to Research Agent
[RESEARCH AGENT] Thought: I need facts about -> 'What is the effect of the demonitisation in our economy how did it affect the small businesses'. Calling the web_search tool
[RESEARCH AGENT] Tool called: web_search_tool
[RESEARCH AGENT] Tool return: 
- The small scale businesses are adversely affected due to this because the whole process of their business is dependent and revolves around the
- Many reports stated that small traders have immensely affected after demonetization because of the cash crunch and lack of infrastructure like digital payment
- It aimed to curb black money,

## 8. Turn It Into a Chatbot (while loop)

In [ ]:
# HINT: EXIT_WORDS = {"exit", "quit", "bye"}
# HINT: print a welcome message

# HINT: while True:
#       - wrap input("You: ") in try/except so the cell exits cleanly if no
#         keyboard is attached (e.g. an automated "run all cells")
#       - skip empty input with continue
#       - break out of the loop if the user typed an exit word
#       - build a FRESH chat_state (matching AgentState) every single loop -
#         remember, this version has no memory between turns
#       - result = graph.invoke(chat_state)
#       - print the bot's result["final_output"]

EXIT_WORDS = {"exit", "quit", "bye"}

while True:
    try:
        user_query = input("You: ").strip()
    except Exception:
        print("No interactive input available here - stopping the chatbot loop.")
        break

    if not user_query:
        continue

    if user_query.lower() in EXIT_WORDS:
        print("Bot: Goodbye!")
        break

    chat_state: AgentState = {
        "query": user_query,
        "research_data": "",
        "final_output": ""
    }

    result = graph.invoke(chat_state)

    print(f"\nBot: {result['final_output']}\n")

[SUPERVISOR] Thought: I received the query -> 'What is the current interest rate?'.I will decide wheather this needs research, but I will not answer it myself.
[SUPERVISOR] Decision: this looks like a factual question -> route to Research Agent
[RESEARCH AGENT] Thought: I need facts about -> 'What is the current interest rate?'. Calling the web_search tool
[RESEARCH AGENT] Tool called: web_search_tool
[RESEARCH AGENT] Tool return: 
- Today's competitive mortgage rates ; 30-year fixed. Rate 6.625%. APR 6.833% ; 20-year fixed. Rate 6.375%. APR 6.680% ; 15-year fixed. Rate 5.875%. APR 6.223% ; 10y/
- Today's Rocket Mortgage® rates ; 30-year fixed · 6.75% · 7.027% ; 30-year FHA · 5.875% · 6.712% ; 20-year fixed · 6.5% · 6.89% ; 15-year fixed · 5.875% · 6.33% ; 30-year VA
- Today's 30-year fixed mortgage rates ; Conventional fixed-rate loans · 30-year. 6.375%. 6.547%. $2,526 ; Conforming adjustable-rate mortgage (ARM) loans · 10/6 mo.
- Selected Interest Rates · 1-year, 3.96, 3.94, 3.97, 3.

## Next Steps (for when we "improvise")

- Swap `web_search_tool`'s mock fallback for a real search API.
- Make the Supervisor's routing decision with the LLM instead of a keyword list.
- Add a `Critique` node that checks the Writer Agent's answer.
- Give the chatbot real memory across turns (without changing `AgentState`'s 3 fields).